In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import f_oneway, chi2_contingency
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('images', exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [3]:
campaigns = pd.read_csv('data/campaigns.csv')
customers = pd.read_csv('data/customers.csv')
purchases = pd.read_csv('data/purchases.csv')
sessions = pd.read_csv('data/website_sessions.csv')
ab_test = pd.read_csv('data/ab_test.csv')

print("All datasets loaded.")

All datasets loaded.


Hypothesis 1 - Platform ROI (Google Ads vs LinkedIn)

In [4]:
# Get ROI for each platform
google_roi = campaigns[campaigns['PlatformID'] == 1]['ROI']
linkedin_roi = campaigns[campaigns['PlatformID'] == 7]['ROI']

# Calculate statistics
google_mean = google_roi.mean()
linkedin_mean = linkedin_roi.mean()
diff = google_mean - linkedin_mean

# Welch's t-test
t_stat, p_value = stats.ttest_ind(google_roi, linkedin_roi, equal_var=False)

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(google_roi)-1) * google_roi.std()**2 + 
                       (len(linkedin_roi)-1) * linkedin_roi.std()**2) / 
                      (len(google_roi) + len(linkedin_roi) - 2))
cohens_d = diff / pooled_std

print("Hypothesis 1: Platform ROI (Google Ads vs LinkedIn)")
print("-" * 40)
print(f"Google Ads ROI: {google_mean:.2f}%")
print(f"LinkedIn ROI: {linkedin_mean:.2f}%")
print(f"Difference: {diff:.2f} percentage points")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Cohen's d: {cohens_d:.3f}")

if p_value < 0.05:
    print("Result: Reject H0. Google Ads ROI is significantly higher than LinkedIn.b")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 1: Platform ROI (Google Ads vs LinkedIn)
----------------------------------------
Google Ads ROI: 206.48%
LinkedIn ROI: 147.84%
Difference: 58.64 percentage points
t-statistic: 13.1374
p-value: 0.000000
Cohen's d: 0.495
Result: Reject H0. Google Ads ROI is significantly higher than LinkedIn.


Hypothesis 2 - Device Conversion (Mobile vs Desktop)

In [5]:
# Get conversion data
mobile_conv = sessions[sessions['Device'] == 'Mobile']['Converted'].sum()
mobile_total = len(sessions[sessions['Device'] == 'Mobile'])
desktop_conv = sessions[sessions['Device'] == 'Desktop']['Converted'].sum()
desktop_total = len(sessions[sessions['Device'] == 'Desktop'])

mobile_rate = mobile_conv / mobile_total
desktop_rate = desktop_conv / desktop_total
diff = desktop_rate - mobile_rate

# Two-proportion z-test
counts = [mobile_conv, desktop_conv]
nobs = [mobile_total, desktop_total]
z_stat, p_value = proportions_ztest(counts, nobs)

# Confidence interval
se = np.sqrt(mobile_rate*(1-mobile_rate)/mobile_total + desktop_rate*(1-desktop_rate)/desktop_total)
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print("Hypothesis 2: Device Conversion (Mobile vs Desktop)")
print("-" * 40)
print(f"Mobile conversion: {mobile_rate*100:.2f}%")
print(f"Desktop conversion: {desktop_rate*100:.2f}%")
print(f"Difference: {diff*100:.2f} percentage points")
print(f"95% CI: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"z-statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Result: Reject H0. Desktop conversion is significantly higher than mobile.")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 2: Device Conversion (Mobile vs Desktop)
----------------------------------------
Mobile conversion: 10.96%
Desktop conversion: 13.10%
Difference: 2.15 percentage points
95% CI: [1.53%, 2.77%]
z-statistic: -6.8856
p-value: 0.000000
Result: Reject H0. Desktop conversion is significantly higher than mobile.


Hypothesis 3 - Weekend vs Weekday Session Duration

In [6]:
# Get session durations
weekend = sessions[sessions['IsWeekend'] == 1]['SessionDuration']
weekday = sessions[sessions['IsWeekend'] == 0]['SessionDuration']

# Calculate statistics
weekend_mean = weekend.mean()
weekday_mean = weekday.mean()
diff = weekend_mean - weekday_mean

# Welch's t-test
t_stat, p_value = stats.ttest_ind(weekend, weekday, equal_var=False)

# Confidence interval
se = np.sqrt(weekend.var()/len(weekend) + weekday.var()/len(weekday))
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print("Hypothesis 3: Weekend vs Weekday Session Duration")
print("-" * 40)
print(f"Weekend duration: {weekend_mean:.1f} seconds")
print(f"Weekday duration: {weekday_mean:.1f} seconds")
print(f"Difference: {diff:.1f} seconds")
print(f"95% CI: [{ci_lower:.1f}, {ci_upper:.1f}]")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0. Weekend sessions differ significantly.")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 3: Weekend vs Weekday Session Duration
----------------------------------------
Weekend duration: 178.5 seconds
Weekday duration: 180.3 seconds
Difference: -1.8 seconds
95% CI: [-4.2, 0.6]
t-statistic: -1.4356
p-value: 0.1511
Result: Fail to reject H0. No significant difference.


Hypothesis 4 - Platform ROI (One-Way ANOVA)

In [13]:
# Group ROI by platform
platform_groups = [campaigns[campaigns['PlatformID'] == pid]['ROI'] for pid in range(1, 8)]

# ANOVA
f_stat, p_value = f_oneway(*platform_groups)

# Effect size (Eta-squared)
ss_between = sum(len(g) * (g.mean() - campaigns['ROI'].mean())**2 for g in platform_groups)
ss_total = sum((campaigns['ROI'] - campaigns['ROI'].mean())**2)
eta_squared = ss_between / ss_total

print("Hypothesis 4: Platform ROI (One-Way ANOVA)")
print("-" * 40)

# Show platform means
platform_names = ['Google Ads', 'TikTok', 'Instagram', 'YouTube', 'Facebook', 'X', 'LinkedIn']
print("Platform ROI means:")
for name, group in zip(platform_names, platform_groups):
    print(f"  {name}: {group.mean():.1f}%")

print(f"\nF-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Eta-squared: {eta_squared:.3f}")

if p_value < 0.05:
    print("Result: Reject H0. Platform significantly affects ROI.")
    
    # Tukey HSD post-hoc test
    platform_data = campaigns[['PlatformID', 'ROI']].copy()
    platform_data['Platform'] = platform_data['PlatformID'].map({
        1:'Google Ads', 2:'TikTok', 3:'Instagram', 4:'YouTube',
        5:'Facebook', 6:'X', 7:'LinkedIn'
    })
    
    tukey = pairwise_tukeyhsd(platform_data['ROI'], platform_data['Platform'], alpha=0.05)
    
    print("\nSignificant pairwise differences:")
    
    # Get the Tukey summary as a list
    tukey_summary = tukey.summary()
    
    # Extract data from the summary table
    # The first row contains column headers, so we skip it
    for row in tukey_summary[1:]:
        # Each row is a tuple with 7 elements: (group1, group2, meandiff, p-adj, lower, upper, reject)
        # Convert Cell objects to strings first, then to appropriate types
        group1 = str(row[0])
        group2 = str(row[1])
        meandiff = float(str(row[2]))
        p_adj = float(str(row[3]))
        reject = row[6]  # This is a boolean
        
        if reject:
            print(f"  {group1} vs {group2}: diff={meandiff:.1f}% (p_adj={p_adj:.4f})")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 4: Platform ROI (One-Way ANOVA)
----------------------------------------
Platform ROI means:
  Google Ads: 206.5%
  TikTok: 192.5%
  Instagram: 184.4%
  YouTube: 177.6%
  Facebook: 167.2%
  X: 157.6%
  LinkedIn: 147.8%

F-statistic: 35.0793
p-value: 0.000000
Eta-squared: 0.021
Result: Reject H0. Platform significantly affects ROI.

Significant pairwise differences:
  Facebook vs Google Ads: diff=39.3% (p_adj=0.0000)
  Facebook vs Instagram: diff=17.2% (p_adj=0.0072)
  Facebook vs LinkedIn: diff=-19.4% (p_adj=0.0012)
  Facebook vs TikTok: diff=25.3% (p_adj=0.0000)
  Facebook vs X: diff=-9.6% (p_adj=0.4265)
  Facebook vs YouTube: diff=10.4% (p_adj=0.3305)
  Google Ads vs Instagram: diff=-22.1% (p_adj=0.0001)
  Google Ads vs LinkedIn: diff=-58.6% (p_adj=0.0000)
  Google Ads vs TikTok: diff=-14.0% (p_adj=0.0612)
  Google Ads vs X: diff=-48.9% (p_adj=0.0000)
  Google Ads vs YouTube: diff=-28.9% (p_adj=0.0000)
  Instagram vs LinkedIn: diff=-36.6% (p_adj=0.0000)
  Instagram vs TikT

 Hypothesis 5 - Campaign Type Conversion (One-Way ANOVA)

In [16]:
# Group conversion by campaign type
type_groups = [campaigns[campaigns['CampaignTypeID'] == ctid]['ConversionRate'] for ctid in range(1, 7)]

# ANOVA
f_stat, p_value = f_oneway(*type_groups)

# Effect size
ss_between = sum(len(g) * (g.mean() - campaigns['ConversionRate'].mean())**2 for g in type_groups)
ss_total = sum((campaigns['ConversionRate'] - campaigns['ConversionRate'].mean())**2)
eta_squared = ss_between / ss_total

print("Hypothesis 5: Campaign Type Conversion (One-Way ANOVA)")
print("-" * 40)

# Show type means
type_names = ['Search', 'Display', 'Social', 'Video', 'Email', 'Retargeting']
print("Campaign type conversion rates:")
for name, group in zip(type_names, type_groups):
    print(f"  {name}: {group.mean()*100:.2f}%")

print(f"\nF-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Eta-squared: {eta_squared:.3f}")

if p_value < 0.05:
    print("Result: Reject H0. Campaign type significantly affects conversion.")
    
    # Tukey HSD post-hoc test
    type_data = campaigns[['CampaignTypeID', 'ConversionRate']].copy()
    type_data['Type'] = type_data['CampaignTypeID'].map({
        1:'Search', 2:'Display', 3:'Social', 4:'Video', 5:'Email', 6:'Retargeting'
    })
    
    tukey = pairwise_tukeyhsd(type_data['ConversionRate'], type_data['Type'], alpha=0.05)
    
    print("\nSignificant pairwise differences:")
    tukey_summary = tukey.summary()
    
    for row in tukey_summary[1:]:
        group1 = str(row[0])
        group2 = str(row[1])
        meandiff = float(str(row[2]))
        p_adj = float(str(row[3]))
        reject = row[6]
        
        if reject:
            print(f"  {group1} vs {group2}: diff={meandiff*100:.2f}% (p_adj={p_adj:.4f})")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 5: Campaign Type Conversion (One-Way ANOVA)
----------------------------------------
Campaign type conversion rates:
  Search: 6.40%
  Display: 6.04%
  Social: 5.66%
  Video: 5.02%
  Email: 4.28%
  Retargeting: 3.59%

F-statistic: 3733.6727
p-value: 0.000000
Eta-squared: 0.651
Result: Reject H0. Campaign type significantly affects conversion.

Significant pairwise differences:
  Display vs Email: diff=-1.76% (p_adj=0.0000)
  Display vs Retargeting: diff=-2.45% (p_adj=0.0000)
  Display vs Search: diff=0.36% (p_adj=0.0000)
  Display vs Social: diff=-0.37% (p_adj=0.0000)
  Display vs Video: diff=-1.02% (p_adj=0.0000)
  Email vs Retargeting: diff=-0.69% (p_adj=0.0000)
  Email vs Search: diff=2.12% (p_adj=0.0000)
  Email vs Social: diff=1.39% (p_adj=0.0000)
  Email vs Video: diff=0.74% (p_adj=0.0000)
  Retargeting vs Search: diff=2.81% (p_adj=0.0000)
  Retargeting vs Social: diff=2.07% (p_adj=0.0000)
  Retargeting vs Video: diff=1.43% (p_adj=0.0000)
  Search vs Social: diff=-0.74

Hypothesis 6 - Customer Segment CLV (One-Way ANOVA)

In [17]:
# Group CLV by segment
segment_groups = [
    customers[customers['CustomerSegment'] == 'Bronze']['CustomerLifetimeValue'],
    customers[customers['CustomerSegment'] == 'Silver']['CustomerLifetimeValue'],
    customers[customers['CustomerSegment'] == 'Gold']['CustomerLifetimeValue'],
    customers[customers['CustomerSegment'] == 'Platinum']['CustomerLifetimeValue']
]

# ANOVA
f_stat, p_value = f_oneway(*segment_groups)

# Effect size
ss_between = sum(len(g) * (g.mean() - customers['CustomerLifetimeValue'].mean())**2 for g in segment_groups)
ss_total = sum((customers['CustomerLifetimeValue'] - customers['CustomerLifetimeValue'].mean())**2)
eta_squared = ss_between / ss_total

print("Hypothesis 6: Customer Segment CLV (One-Way ANOVA)")
print("-" * 40)

# Show segment means
segments = ['Bronze', 'Silver', 'Gold', 'Platinum']
print("CLV by segment:")
for name, group in zip(segments, segment_groups):
    print(f"  {name}: ${group.mean():,.2f}")

print(f"\nF-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Eta-squared: {eta_squared:.3f}")

if p_value < 0.05:
    print("Result: Reject H0. Segment significantly affects CLV.")
    
    # Tukey HSD post-hoc test
    seg_data = customers[['CustomerSegment', 'CustomerLifetimeValue']].copy()
    
    tukey = pairwise_tukeyhsd(seg_data['CustomerLifetimeValue'], seg_data['CustomerSegment'], alpha=0.05)
    
    print("\nSignificant pairwise differences:")
    tukey_summary = tukey.summary()
    
    for row in tukey_summary[1:]:
        group1 = str(row[0])
        group2 = str(row[1])
        meandiff = float(str(row[2]))
        p_adj = float(str(row[3]))
        reject = row[6]
        
        if reject:
            print(f"  {group1} vs {group2}: diff=${meandiff:,.2f} (p_adj={p_adj:.4f})")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 6: Customer Segment CLV (One-Way ANOVA)
----------------------------------------
CLV by segment:
  Bronze: $4,995.08
  Silver: $9,212.21
  Gold: $21,134.19
  Platinum: $36,835.81

F-statistic: 1284.8847
p-value: 0.000000
Eta-squared: 0.436
Result: Reject H0. Segment significantly affects CLV.

Significant pairwise differences:
  Bronze vs Gold: diff=$16,139.11 (p_adj=0.0000)
  Bronze vs Platinum: diff=$31,840.73 (p_adj=0.0000)
  Bronze vs Silver: diff=$4,217.13 (p_adj=0.0000)
  Gold vs Platinum: diff=$15,701.62 (p_adj=0.0000)
  Gold vs Silver: diff=$-11,921.98 (p_adj=0.0000)
  Platinum vs Silver: diff=$-27,623.60 (p_adj=0.0000)


Hypothesis 7 - Segment × Purchase Status (Chi-Square)

In [18]:
# Merge purchases with customer segments
customer_purchases = purchases.merge(customers[['CustomerID', 'CustomerSegment']], on='CustomerID')

# Create contingency table
contingency = pd.crosstab(customer_purchases['CustomerSegment'], customer_purchases['PurchaseStatus'])

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency)

# Effect size (Cramer's V)
n = contingency.sum().sum()
min_dim = min(contingency.shape[0], contingency.shape[1]) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print("Hypothesis 7: Segment × Purchase Status (Chi-Square)")
print("-" * 40)
print("\nContingency Table:")
print(contingency)

print(f"\nChi-square: {chi2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

if p_value < 0.05:
    print("Result: Reject H0. Segment and purchase status are associated.")
else:
    print("Result: Fail to reject H0. Segment and purchase status are independent.")

Hypothesis 7: Segment × Purchase Status (Chi-Square)
----------------------------------------

Contingency Table:
PurchaseStatus   Cancelled  Completed  Pending  Refunded
CustomerSegment                                         
Bronze                  82       1163      146        69
Gold                   135       2310      269       119
Platinum               177       2800      352       182
Silver                  97       1813      200        86

Chi-square: 11.8711
p-value: 0.2207
Cramer's V: 0.020
Result: Fail to reject H0. Segment and purchase status are independent.


Hypothesis 8 - A/B Test

In [12]:
# Get A/B test data
a_data = ab_test[ab_test['Variant'] == 'A']
b_data = ab_test[ab_test['Variant'] == 'B']

a_visitors = a_data['Visitors'].sum()
a_conv = a_data['Conversions'].sum()
b_visitors = b_data['Visitors'].sum()
b_conv = b_data['Conversions'].sum()

a_rate = a_conv / a_visitors
b_rate = b_conv / b_visitors
diff = b_rate - a_rate

# Two-proportion z-test
counts = [a_conv, b_conv]
nobs = [a_visitors, b_visitors]
z_stat, p_value = proportions_ztest(counts, nobs)

# Lift
lift = ((b_rate - a_rate) / a_rate) * 100

# Confidence interval
se = np.sqrt(a_rate*(1-a_rate)/a_visitors + b_rate*(1-b_rate)/b_visitors)
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print("Hypothesis 8: A/B Test")
print("-" * 40)
print(f"Variant A conversions: {a_conv:,} / {a_visitors:,} = {a_rate*100:.3f}%")
print(f"Variant B conversions: {b_conv:,} / {b_visitors:,} = {b_rate*100:.3f}%")
print(f"Absolute lift: {diff*100:.3f} percentage points")
print(f"Relative lift: {lift:.2f}%")
print(f"95% CI: [{ci_lower*100:.3f}%, {ci_upper*100:.3f}%]")
print(f"z-statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Result: Reject H0. Variant B significantly outperforms Variant A.")
else:
    print("Result: Fail to reject H0. No significant difference.")

Hypothesis 8: A/B Test
----------------------------------------
Variant A conversions: 240,902 / 4,028,446 = 5.980%
Variant B conversions: 288,284 / 3,886,072 = 7.418%
Absolute lift: 1.438 percentage points
Relative lift: 24.05%
95% CI: [1.404%, 1.473%]
z-statistic: -80.9873
p-value: 0.000000
Result: Reject H0. Variant B significantly outperforms Variant A.
